In [28]:
import pandas as pd
import numpy as np
from statsmodels.stats.weightstats import DescrStatsW
import warnings
import os
import json
warnings.filterwarnings('ignore')

print("Environment loaded successfully")

# Load processed data from Notebook 1
df = pd.read_csv('../data/processed/brfss_2023_diabetes.csv')

print(f"Loaded: {len(df):,} respondents")
print(f"Variables: {df.shape[1]}")

# Verify key variables
print("\nKey variables status:")
print(f"  has_diabetes: {df['has_diabetes'].notna().sum():,} non-missing")
print(f"  _LLCPWT (weight): {df['_LLCPWT'].notna().sum():,} non-missing")

# Survey Weight Distribution
print("\nSurvey Weight Statistics:")
print(df['_LLCPWT'].describe())
print(f"\nSum of weights: {df['_LLCPWT'].sum():,.0f}")

# Weighted Prevalence Function
def weighted_prevalence(data, outcome_var, weight_var='_LLCPWT'):
    """
    Calculate weighted prevalence with 95% confidence interval
    """
    valid = data[[outcome_var, weight_var]].dropna()
    
    if len(valid) == 0:
        return {
            'prevalence': np.nan,
            'ci_lower': np.nan,
            'ci_upper': np.nan,
            'n': 0
        }
    
    weighted_stats = DescrStatsW(valid[outcome_var], weights=valid[weight_var])
    prevalence = weighted_stats.mean
    ci_lower, ci_upper = weighted_stats.tconfint_mean()
    
    return {
        'prevalence': prevalence,
        'ci_lower': max(0, ci_lower),
        'ci_upper': min(1, ci_upper),
        'n': len(valid)
    }

print("\nWeighted prevalence function defined")

# Overall Diabetes Prevalence - Unweighted (BIASED)
unweighted = df['has_diabetes'].mean()
print(f"\nUnweighted prevalence: {unweighted:.2%}")
print("  (BIASED - ignores sampling design)")

# Weighted (CORRECT)
result = weighted_prevalence(df, 'has_diabetes')
print(f"\nWeighted prevalence: {result['prevalence']:.2%}")
print(f"95% CI: [{result['ci_lower']:.2%}, {result['ci_upper']:.2%}]")
print(f"Sample size: {result['n']:,}")
print("  (CORRECT - accounts for survey design)")

# Calculate bias
bias = unweighted - result['prevalence']
print(f"\nBias from ignoring weights: {bias:.2%}")

# Prevalence by Age Group
age_labels = {
    1: '18-24',
    2: '25-34',
    3: '35-44',
    4: '45-54',
    5: '55-64',
    6: '65+'
}

age_results = []
for age in sorted(df['_AGE_G'].dropna().unique()):
    subset = df[df['_AGE_G'] == age]
    res = weighted_prevalence(subset, 'has_diabetes')
    
    age_results.append({
        'age_group': age_labels.get(age, str(age)),
        'prevalence': res['prevalence'],
        'ci_lower': res['ci_lower'],
        'ci_upper': res['ci_upper'],
        'n': res['n']
    })

age_df = pd.DataFrame(age_results)

print("\nDIABETES PREVALENCE BY AGE GROUP")
print("="*70)
for _, row in age_df.iterrows():
    print(f"{row['age_group']:<10} {row['prevalence']:>6.1%}  [{row['ci_lower']:.1%}, {row['ci_upper']:.1%}]  n={row['n']:,}")

# Prevalence by Sex
sex_results = []
for sex in [1, 2]:
    subset = df[df['BIRTHSEX'] == sex]
    res = weighted_prevalence(subset, 'has_diabetes')
    
    sex_label = 'Male' if sex == 1 else 'Female'
    sex_results.append({
        'sex': sex_label,
        'prevalence': res['prevalence'],
        'ci_lower': res['ci_lower'],
        'ci_upper': res['ci_upper'],
        'n': res['n']
    })

sex_df = pd.DataFrame(sex_results)

print("\nDIABETES PREVALENCE BY SEX")
print("="*70)
for _, row in sex_df.iterrows():
    print(f"{row['sex']:<10} {row['prevalence']:>6.1%}  [{row['ci_lower']:.1%}, {row['ci_upper']:.1%}]  n={row['n']:,}")

# Prevalence by Race/Ethnicity
race_labels = {
    1: 'White',
    2: 'Black',
    3: 'Asian',
    4: 'Native American',
    5: 'Hispanic',
    6: 'Other',
    7: 'Multiracial',
    8: 'Other'
}

race_results = []
for race in sorted(df['_RACE'].dropna().unique()):
    subset = df[df['_RACE'] == race]
    res = weighted_prevalence(subset, 'has_diabetes')
    
    race_results.append({
        'race': race_labels.get(race, 'Unknown'),
        'prevalence': res['prevalence'],
        'ci_lower': res['ci_lower'],
        'ci_upper': res['ci_upper'],
        'n': res['n']
    })

race_df = pd.DataFrame(race_results).sort_values('prevalence', ascending=False)

print("\nDIABETES PREVALENCE BY RACE/ETHNICITY")
print("="*70)
for _, row in race_df.iterrows():
    print(f"{row['race']:<20} {row['prevalence']:>6.1%}  [{row['ci_lower']:.1%}, {row['ci_upper']:.1%}]  n={row['n']:,}")

# Save Results
output_data = {
    'overall': {
        'prevalence': float(result['prevalence']),
        'ci_lower': float(result['ci_lower']),
        'ci_upper': float(result['ci_upper']),
        'n': int(result['n'])
    },
    'by_age': age_df.to_dict('records'),
    'by_sex': sex_df.to_dict('records'),
    'by_race': race_df.to_dict('records')
}

os.makedirs('../outputs/tables', exist_ok=True)
output_path = '../outputs/tables/weighted_prevalence.json'

with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2, default=str)

print(f"\nSaved results to: {output_path}")

# Final summary
print("\n" + "="*70)
print("NOTEBOOK 2 COMPLETE: SURVEY WEIGHTING")
print("="*70)
print(f"Overall diabetes prevalence: {result['prevalence']:.2%}")
print(f"95% CI: [{result['ci_lower']:.2%}, {result['ci_upper']:.2%}]")
print(f"\nAge range: {age_df['prevalence'].min():.1%} to {age_df['prevalence'].max():.1%}")
print(f"Racial disparity: {race_df['prevalence'].min():.1%} to {race_df['prevalence'].max():.1%}")
print("="*70)

Environment loaded successfully
Loaded: 10,000 respondents
Variables: 25

Key variables status:
  has_diabetes: 10,000 non-missing
  _LLCPWT (weight): 10,000 non-missing

Survey Weight Statistics:
count    10000.000000
mean         1.766899
std          0.722393
min          0.500039
25%          1.135216
50%          1.773782
75%          2.400676
max          2.999973
Name: _LLCPWT, dtype: float64

Sum of weights: 17,669

Weighted prevalence function defined

Unweighted prevalence: 12.31%
  (BIASED - ignores sampling design)

Weighted prevalence: 12.55%
95% CI: [12.06%, 13.04%]
Sample size: 10,000
  (CORRECT - accounts for survey design)

Bias from ignoring weights: -0.24%

DIABETES PREVALENCE BY AGE GROUP
18-24       12.2%  [11.0%, 13.4%]  n=1,663
25-34       12.6%  [11.4%, 13.7%]  n=1,753
35-44       12.9%  [11.7%, 14.1%]  n=1,664
45-54       13.0%  [11.7%, 14.2%]  n=1,623
55-64       12.6%  [11.5%, 13.8%]  n=1,662
65+         12.0%  [10.8%, 13.2%]  n=1,635

DIABETES PREVALENCE BY 